Ridership ≈ Service × Demand × Accessibility

In [1]:
pip install shared_utils

Note: you may need to restart the kernel to use updated packages.


In [2]:
# Importing necessary package 
import pandas as pd 
import geopandas as gpd
pd.set_option('display.max_columns', None)
import os
import google.auth
import gcsfs
fs = gcsfs.GCSFileSystem()
import statsmodels.api as sm
import numpy as np
from scipy.stats import skew


import statsmodels.api as sm
from statsmodels.stats.outliers_influence import variance_inflation_factor

In [3]:
GCS_FILE_PATH = 'gs://calitp-analytics-data/data-analyses/ahsc_grant/ahsc_riderships/AHSC_2026'

In [4]:
with fs.open(f"{GCS_FILE_PATH}/stop_route_df.parquet", "rb") as f: 
    stop_route_df = gpd.read_parquet(f)

In [14]:
stop_route_df.head(2)

,organization_name,feed_key,route_id,route_name,stop_id,stop_name,stop_code,n_trips,n_routes,pt_geom,day_type,average_daily_boardings,average_daily_alightings,start_date,end_date,geometry_x,geometry_y,total_pop_adj,poverty_pop_adj,non_us_citizen_adj,workers_with_no_car_adj,households_with_no_cars_adj,disabled_pop_adj,public_asst_pop_adj,inc_extremelylow_adj,inc_verylow_adj,inc_low_adj,male_seniors_adj,female_seniors_adj,veteran_pop_adj,male_youth_adj,inc_total_lowincome_adj,female_youth_adj,total_seniors_adj,jobs_tot_adj,total_youth_adj,ALAND_adj,is_route_level,n_routes_effective
0,Gold Coast Transit,3cb676436aad669e52042c0e97a9a051,16,None,VNACLR1,.,None,32.0,1.0,POINT(-119.294028 34.343645),Weekday,0.0,3.0,2025-05-01,2025-05-31,POINT (64936.582 -407800.219),"POLYGON ((65737.380 -407879.090, 65725.793 -40...",23.421382,2.988619,2.473963,0.334075,0.270872,2.600369,6.898202,4.984041,4.532588,2.121829,1.480766,1.516882,0.659121,0.740383,11.638459,2.094742,2.997648,4.207542,2.835125,1.092105e+06,1,0.0
1,Gold Coast Transit,3cb676436aad669e52042c0e97a9a051,16,None,VNACLR1,.,None,32.0,1.0,POINT(-119.294028 34.343645),Weekday,0.0,3.0,2025-05-01,2025-05-31,POINT (64936.582 -407800.219),"POLYGON ((65015.454 -408601.016, 64936.582 -40...",19.338329,1.617058,0.585279,0.102575,0.241352,3.252218,8.374914,4.434843,5.092527,1.442078,2.455757,2.823818,1.339504,0.959374,10.969448,0.645617,5.279575,2.594534,1.604991,8.775517e+05,1,0.0


In [6]:
stop_route_df['is_route_level'] = stop_route_df['organization_name'].isin([
    'Foothill Transit',
    'Gold Coast Transit',
    'Golden Gate Transit',
    'Long Beach Transit',
    'Orange County Transportation Authority',
    'SacRT Bus',
    'Samtrans',
    'SDMTS',
    'Culver City Bus',
    'Big Blue Bus',
    'Riverside Transit'
]).astype(int)

In [7]:
stop_route_df['n_routes_effective'] = stop_route_df['n_routes'] * (1 - stop_route_df['is_route_level'])

### Log-linear Regression Model

In [8]:
# Log-linear regression
# Dependent variable: log('average_daily_boardings')
# Predictors: n_trips, n_routes, total_pop_adj, households_with_no_cars_adj, jobs_tot_adj

# Copy the dataframe
df = stop_route_df.copy()

# 2. Create log dependent variable
# Replace 0 with NaN BEFORE log (log(0) = -inf)
df['log_boardings'] = np.log(df['average_daily_boardings'].replace(0, np.nan))

# 3. Select only rows with all needed variables
df = df.dropna(subset=[
    'average_daily_boardings',
    'n_trips',
    'n_routes_effective',
    'total_pop_adj',
    'households_with_no_cars_adj'
])

# 4. Define Y and X
y = df['average_daily_boardings']

X = df[['n_trips', 'n_routes_effective',
        'total_pop_adj', 'households_with_no_cars_adj']]

# 5. Add constant
X = sm.add_constant(X)

# 6. Fit model
model = sm.OLS(y, X).fit()

# 7. Show results
print(model.summary())

                               OLS Regression Results                              
Dep. Variable:     average_daily_boardings   R-squared:                       0.268
Model:                                 OLS   Adj. R-squared:                  0.268
Method:                      Least Squares   F-statistic:                 1.383e+04
Date:                     Thu, 26 Mar 2026   Prob (F-statistic):               0.00
Time:                             19:13:53   Log-Likelihood:            -1.1223e+06
No. Observations:                   151044   AIC:                         2.245e+06
Df Residuals:                       151039   BIC:                         2.245e+06
Df Model:                                4                                         
Covariance Type:                 nonrobust                                         
                                  coef    std err          t      P>|t|      [0.025      0.975]
----------------------------------------------------------------

- About 26.8% of the variation in dependent variable (here, stop-level ridership) is explained by the predictors in the model.
- Skew = 28.63 → Residuals are extremely asymmetric, violating normality.
- Kurtosis = 1793.542 → Residuals have ultra‑heavy tails, far from a normal distribution.
- JB statistic = 20 billion → The Jarque–Bera test overwhelmingly rejects normality.
- Durbin–Watson = 0.231 → Strong positive autocorrelation remains in the residuals.
- Condition number = 1.22e+04 → Predictors have scaling issues and possible multicollinearity.

### Checking Multicollinearity using VIF

VIF checks multicollinearity (correlation among predictors)

- Idea:
If a variable can be well explained by other predictors,
its coefficient becomes unstable and its standard error increases
- Interpretation:
VIF ≈ 1   → no correlation (good);
VIF > 5   → moderate concern;
VIF > 10  → serious multicollinearity
- Key point:
High VIF doesn't bias coefficients, but makes them noisy and hard to interpret

In [9]:
# Selecting only numeric regressors (Ridership ≈ Service × Demand × Accessibility)
X = stop_route_df[
    ['n_trips', 'n_routes_effective',  'total_pop_adj', 'households_with_no_cars_adj' ]
].copy()

# Add constant
X = sm.add_constant(X)

vif_df = pd.DataFrame()
vif_df["variable"] = X.columns
vif_df["VIF"] = [
    variance_inflation_factor(X.values, i)
    for i in range(X.shape[1])
]

print(vif_df)

                      variable       VIF
0                        const  3.381647
1                      n_trips  1.157433
2           n_routes_effective  1.153209
3                total_pop_adj  1.688035
4  households_with_no_cars_adj  1.686701


All predictors have low VIFs (≈1–1.8), which means very little multicollinearity among chosen variables.
The model’s coefficients should therefore be stable and interpretable, with no inflation of standard errors.
The high VIF on the constant is not meaningful and can be ignored.

In [10]:
stop_route_df['average_daily_boardings'].describe()

count    151044.000000
mean         51.376269
std         477.085033
min           0.000000
25%           1.590909
50%           6.285714
75%          20.531565
max       41893.868537
Name: average_daily_boardings, dtype: float64

Data is highly skewed (28.63) .
Skewed data is data that isn’t evenly distributed around the center, values bunch up on one side and stretch into a long tail.
This means a few stops have very high boardings and push the distribution out into a long tail.
75% of stops board ≤ 20 people

Possible solution: Use a count model (Poisson or Negative Binomial)  
Other models to check:
- Poisson
- Negative Binomial (handles overdispersion better)
- Zero‑inflated Poisson (if many zeros)
- Zero‑inflated NB (most flexible)

### Poisson Model

In [11]:
# Copy the dataset
df = stop_route_df.copy()

# 2. Select needed variables (drop missing)
df = df.dropna(subset=[
    'average_daily_boardings',
    'n_trips', 'n_routes_effective',
    'total_pop_adj', 'households_with_no_cars_adj'
])

# 3. Define Y and X
y = df['average_daily_boardings']

X = df[['n_trips', 'n_routes_effective',
        'total_pop_adj', 'households_with_no_cars_adj']]

# 4. Add intercept
X = sm.add_constant(X)

# 5. Fit Poisson GLM
poisson_model = sm.GLM(y, X, family=sm.families.Poisson()).fit()

# 6. Output summary
print(poisson_model.summary())


                    Generalized Linear Model Regression Results                    
Dep. Variable:     average_daily_boardings   No. Observations:               151044
Model:                                 GLM   Df Residuals:                   151039
Model Family:                      Poisson   Df Model:                            4
Link Function:                         Log   Scale:                          1.0000
Method:                               IRLS   Log-Likelihood:            -9.3019e+06
Date:                     Thu, 26 Mar 2026   Deviance:                   1.8065e+07
Time:                             19:13:54   Pearson chi2:                 8.65e+07
No. Iterations:                          8   Pseudo R-squ. (CS):              1.000
Covariance Type:                 nonrobust                                         
                                  coef    std err          z      P>|z|      [0.025      0.975]
----------------------------------------------------------------

This Poisson model fits very poorly: the Pearson chi‑square is huge, meaning the data are heavily over‑dispersed and the Poisson assumptions break.
- The coefficient for n_trips (0.0163) means each extra trip increases expected ridership by about 1.6%.
- n_routes (–0.268) is negative, which is unrealistic and indicates model misspecification due to over‑dispersion.
- total_pop_adj is negative, also unrealistic — another sign the Poisson model is not appropriate.
- households_with_no_cars_adj is positive, meaning more zero‑car households predict higher ridership.
Overall: the Poisson model’s signs are distorted.

Pearson chi‑square = 4.77e+07 and Degrees of freedom ≈ 143325.
Hence, 
Dispersion estimate = Pearson χ² / df; 8.30e+07 / 151,044 ≈ 332.8

A valid Poisson model should have dispersion ≈ 1. Dispersion ≈ 550, which means:
Poisson model is severely overdispersed. >1 → overdispersion (variance is larger than the model assumes).

Variance >> mean:
- Poisson standard errors become too small
- p‑values become artificially tiny
- coefficients look falsely “precise”
- model fit is misleading

### Negative Binomial Model 

In [12]:
# Same X and y as  Poisson
nb_model = sm.GLM(
    y,
    X,
    family=sm.families.NegativeBinomial()
).fit()

print(nb_model.summary())

/opt/conda/lib/python3.11/site-packages/statsmodels/genmod/families/family.py:1367: ValueWarning: Negative binomial dispersion parameter alpha not set. Using default value alpha=1.0.
  warnings.warn("Negative binomial dispersion parameter alpha not "


                    Generalized Linear Model Regression Results                    
Dep. Variable:     average_daily_boardings   No. Observations:               151044
Model:                                 GLM   Df Residuals:                   151039
Model Family:             NegativeBinomial   Df Model:                            4
Link Function:                         Log   Scale:                          1.0000
Method:                               IRLS   Log-Likelihood:            -5.7523e+05
Date:                     Thu, 26 Mar 2026   Deviance:                   2.7721e+05
Time:                             19:13:56   Pearson chi2:                 8.17e+05
No. Iterations:                         16   Pseudo R-squ. (CS):             0.8978
Covariance Type:                 nonrobust                                         
                                  coef    std err          z      P>|z|      [0.025      0.975]
----------------------------------------------------------------

The Negative Binomial model behaves much better than the Poisson model because it corrects the strongly overdispersed nature of ridership data. After switching to a Negative Binomial model, the dispersion dropped to a reasonable level (≈5), the standard errors became realistic, the z‑values returned to interpretable ranges, and the overall fit stabilized. 


The NB model is used when:
- counts vary wildly
- some stops have way more riders than others
- data is highly skewed
- there are outliers or “hot spots”

Variance > Mean
This matches real transit ridership perfectly.

In the above model,
Pseudo R² (Cragg‑Uhler / Nagelkerke): 0.8978
Very high — the predictors explain a large proportion of the variation in ridership.
For transit ridership data (often noisy and over‑dispersed), this is exceptionally strong.



### Variables Interpretation

1. n_trips coefficient = 0.0245
Interpretation: Each additional trip serving the stop increases expected daily boardings by:
exp(0.0245) − 1 ≈ 2.48%

2. n_routes_effective coefficient = 0.3380
Interpretation: For stops/agencies where n_routes_effective > 0 (aggregated stop-level data), each additional route increases expected daily boardings by:
exp(0.3380)−1 ≈ 40.2%
Important: This coefficient only applies to stops/organizations where aggregated route data exists.
Route-level agencies (is_route_level = 1) have n_routes_effective = 0, so this coefficient does not affect them.



3. total_pop_adj – Population in stop buffer
total_pop_adj represents the estimated number of residents physically inside a stop’s 600 m buffer, after adjusting each census tract by the fraction of its area that overlaps the buffer.
Example: if 10% of a tract lies inside a stop’s buffer, 10% of the tract’s population is assigned to that stop.
Coefficient interpretation:
Model coefficient: 2.296e-05
Effect per additional person: exp(2.296𝑒−05)−1≈0.0023% increase in expected daily boardings per person

  This looks tiny for one person, but stop buffers usually contain hundreds or thousands of people.
     Scaling to realistic population changes:

    +1,000 residents in the stop buffer: exp(1000×2.296𝑒−05)−1 ≈ 2.32% increase in ridership
    +10,000 residents in the stop buffer: exp(10,000×2.296𝑒−05)−1 ≈ 25.8% increase in ridership

4. households_with_no_cars_adj (coef = 4.162e-05)
households_with_no_cars_adj is the estimated number of zero‑car households within the stop catchment, computed the same area-weighted way.

For each additional zero-car household in the catchment:
exp(4.162e-05) – 1 ≈ 0.00416% increase in expected boardings

- +100 zero-car households in buffer : ≈ 0.42% ridership increase
- +1,000 zero-car households : ≈ 4.2% ridership increase

#### Adding more variables in the model

In [15]:
X_extra = df[['n_trips', 'n_routes_effective',
        'total_pop_adj', 'households_with_no_cars_adj',  'total_youth_adj',  'total_seniors_adj']]

# Same X and y as  Poisson
nb_model_extended = sm.GLM(
    y,
    X_extra,
    family=sm.families.NegativeBinomial()
).fit()

print(nb_model_extended.summary())

/opt/conda/lib/python3.11/site-packages/statsmodels/genmod/families/family.py:1367: ValueWarning: Negative binomial dispersion parameter alpha not set. Using default value alpha=1.0.
  warnings.warn("Negative binomial dispersion parameter alpha not "


                    Generalized Linear Model Regression Results                    
Dep. Variable:     average_daily_boardings   No. Observations:               151044
Model:                                 GLM   Df Residuals:                   151038
Model Family:             NegativeBinomial   Df Model:                            5
Link Function:                         Log   Scale:                          1.0000
Method:                               IRLS   Log-Likelihood:            -5.9748e+05
Date:                     Thu, 26 Mar 2026   Deviance:                   3.2170e+05
Time:                             19:26:30   Pearson chi2:                 1.26e+06
No. Iterations:                         20   Pseudo R-squ. (CS):             0.8628
Covariance Type:                 nonrobust                                         
                                  coef    std err          z      P>|z|      [0.025      0.975]
----------------------------------------------------------------

## Negative Binomial GLM (Log‑Link) Model Equation

The model estimates the expected daily boardings at stop *i* using a Negative Binomial
Generalized Linear Model with a log link.

The linear predictor is:

$$
\log(\mu_i) =
\beta_0
+ \beta_1 \, (\text{n\_trips}_i)
+ \beta_2 \, (\text{n\_routes}_i)
+ \beta_3 \, (\text{total\_pop\_adj}_i)
+ \beta_4 \, (\text{households\_with\_no\_cars\_adj}_i)
+ \beta_5 \, (\text{total\_youth\_adj}_i)
+ \beta_6 \, (\text{inc\_total\_lowincome\_adj}_i)
+ \beta_7 \, (\text{total\_seniors\_adj}_i)
$$

Where the expected value of daily boardings is:

$$
\mu_i = E[\text{average\_daily\_boardings}_i]
$$

The outcome follows a Negative Binomial distribution:

$$
Y_i \sim \text{NB}(\mu_i, \alpha)
$$

The log link implies that predictor effects are multiplicative:

$$
\mu_i = \exp\left(
\beta_0
+ \beta_1 \, \text{n\_trips}_i
+ \beta_2 \, \text{n\_routes}_i
+ \beta_3 \, \text{total\_pop\_adj}_i
+ \beta_4 \, \text{households\_with\_no\_cars\_adj}_i
+ \beta_5 \, \text{total\_youth\_adj}_i
+ \beta_6 \, \text{inc\_total\_lowincome\_adj}_i
+ \beta_7 \, \text{total\_seniors\_adj}_i
\right)
$$

Statsmodels’ `GLM(..., family=NegativeBinomial())` uses a fixed dispersion parameter $\alpha$
unless otherwise specified.

In log_linear ols regression model Y is continuous variable i.e. if n_trip has coefficient of 0.1 it means A one‑unit increase in trips increases the geometric mean of ridership by 10%.
While in this model: it means Each additional daily trip increases expected ridership by about 10%.

1) Log-linear OLS model
Modeling log(ridership). 
When the coefficient is 0.1, it means:
→ adding 1 trip increases ridership by about 10% on average (geometric mean).
2) Negative Binomial model
This model is built for count data (like number of riders).
A coefficient of 0.1 also means about a 10% increase, but interpreted as:
→ each extra trip increases the expected number of riders by 10%.

They sound almost identical, but:
- Log-linear OLS focuses on modeling a transformed version of ridership (log scale). Talks about the geometric mean (a bit abstract)
Negative Binomial
- directly models counts (actual riders). Talks about expected ridership (more natural and realistic for counts)